# Module 1: LangGraph Fundamentals

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **StateGraph**: the core LangGraph building block
- **TypedDict state**: typed state that flows through the graph
- **Nodes**: Python functions that update state
- **Edges**: connections between nodes (normal and conditional)
- **START/END**: entry and exit points
- **Conditional routing**: directing flow based on state values

## Core mental model
```
STATE (TypedDict) --> NODE (function) --> updated STATE --> next NODE
```
Every LangGraph application follows this pattern.


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. BMI Calculator — StateGraph with Conditional Routing

This graph demonstrates the full LangGraph pattern:
- `BMIState` TypedDict flows through the graph
- Each node receives state, returns a partial update dict
- Conditional edge routes to different recommendation nodes

```
START -> calculate_bmi -> classify_bmi -> [underweight|normal|overweight|obese] -> END
```

In [ ]:
from day4.langgraph_basics import build_bmi_graph, run_graph, inspect_graph

bmi_graph = build_bmi_graph()

# Inspect the graph structure
info = inspect_graph(bmi_graph)
print(f'Graph nodes: {info["nodes"]}')
print(f'Total nodes: {info["num_nodes"]}')

In [ ]:
# Run BMI graph for different cases
test_cases = [
    {'weight_kg': 55, 'height_m': 1.70, 'bmi': None, 'category': None, 'recommendation': None},
    {'weight_kg': 70, 'height_m': 1.75, 'bmi': None, 'category': None, 'recommendation': None},
    {'weight_kg': 90, 'height_m': 1.70, 'bmi': None, 'category': None, 'recommendation': None},
    {'weight_kg': 110, 'height_m': 1.65, 'bmi': None, 'category': None, 'recommendation': None},
]

for case in test_cases:
    result = run_graph(bmi_graph, case)
    print(f'{result["weight_kg"]}kg / {result["height_m"]}m -> BMI={result["bmi"]} ({result["category"]})')
    print(f'  Recommendation: {result["recommendation"][:70]}')

## 2. Prompt Chaining Graph

Chain three sequential steps: **report -> summary -> tweet**.

Each node passes its output to the next via the shared state.
This pattern is used by newsrooms, content agencies, and marketing teams.

In [ ]:
from day4.langgraph_basics import build_prompt_chain_graph

# Mock LLM — replace with ChatOpenAI for real use
def mock_llm(prompt: str) -> str:
    if 'report' in prompt.lower():
        return 'India AI market is growing rapidly. Key players: TCS, Infosys, Wipro. Government initiatives like AI Mission are accelerating adoption across sectors.'
    elif 'bullet' in prompt.lower() or 'summarise' in prompt.lower():
        return '- India AI market: $6B by 2025\n- Key players: TCS, Infosys, Wipro\n- Gov AI Mission: accelerating adoption'
    else:
        return 'India AI market is booming! $6B by 2025, driven by TCS, Infosys & Gov AI Mission. #IndiaAI #TechIndia'

chain = build_prompt_chain_graph(mock_llm)
result = chain.invoke({'topic': 'India AI market', 'detailed_report': None, 'summary': None, 'tweet': None})

print('Topic:', result['topic'])
print()
print('Report:', result['detailed_report'][:100], '...')
print()
print('Summary:\n', result['summary'])
print()
print('Tweet:', result['tweet'])

## 3. Review Sentiment + Auto-Reply Graph

A conditional routing example that classifies customer reviews and generates
appropriate replies. Used by e-commerce companies (Flipkart, Amazon India, Meesho).

In [ ]:
from day4.langgraph_basics import build_review_graph

review_graph = build_review_graph()

reviews = [
    'This product is amazing and great quality! Love it!',
    'Terrible experience, worst service ever, horrible packaging.',
    'The item arrived in 3 days.',
    'Excellent delivery, fantastic product, best purchase this year!',
]

for review in reviews:
    result = review_graph.invoke({'review_text': review, 'sentiment': None, 'reply': None})
    print(f'[{result["sentiment"].upper():8s}] {review[:50]}...')
    print(f'           Reply: {result["reply"][:80]}')
    print()

## 4. Building Your Own Graph — Template

Use this template to build any LangGraph workflow:

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

# 1. Define your state
class MyState(TypedDict):
    input_text: str
    processed: Optional[str]
    result: Optional[str]

# 2. Define your nodes
def process_node(state: MyState) -> dict:
    return {'processed': state['input_text'].upper()}

def result_node(state: MyState) -> dict:
    return {'result': f'Final: {state["processed"]}'}

# 3. Build and compile
g = StateGraph(MyState)
g.add_node('process', process_node)
g.add_node('result', result_node)
g.add_edge(START, 'process')
g.add_edge('process', 'result')
g.add_edge('result', END)
graph = g.compile()

# 4. Run
output = graph.invoke({'input_text': 'hello world', 'processed': None, 'result': None})
print('Result:', output['result'])

## Databricks Bridge

When running on Databricks, LangGraph graphs can be used inside Spark UDFs or MLflow pipelines:

In [ ]:
# Databricks integration pattern
# In a Databricks notebook:
#
# from pyspark.sql.functions import udf
# from pyspark.sql.types import StringType
#
# bmi_graph = build_bmi_graph()
#
# @udf(StringType())
# def classify_patient_bmi(weight_kg, height_m):
#     result = bmi_graph.invoke({
#         'weight_kg': float(weight_kg),
#         'height_m': float(height_m),
#         'bmi': None, 'category': None, 'recommendation': None
#     })
#     return result['category']
#
# df_with_bmi = patients_df.withColumn('bmi_category', classify_patient_bmi('weight_kg', 'height_m'))

print('Databricks bridge pattern shown above.')
print('LangGraph graphs are pure Python — they run anywhere: local, Databricks, Lambda, Cloud Run.')